# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata is an mlcroissant.DatasetMetadata object
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by @id and fields
if hasattr(metadata, 'record_sets'):
    print('Available Record Sets:')
    for rset in metadata.record_sets:
        print(f"- Record Set: {rset['@id']}")
        if 'fields' in rset:
            print('  Fields:')
            for fld in rset['fields']:
                print(f"    - Field @id: {fld['@id']}, name: {fld.get('name', '')}")
        if 'columns' in rset:
            print('  Columns:')
            for col in rset['columns']:
                print(f"    - Column @id: {col['@id']}, name: {col.get('name', '')}")
else:
    print('No record sets found in metadata!')

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from found record sets

# Get the list of available record set @id's from the metadata
if hasattr(metadata, 'record_sets'):
    record_sets = [rset['@id'] for rset in metadata.record_sets]
    print('Record sets to load:', record_sets)
else:
    record_sets = []

dataframes = {}

for record_set_id in record_sets:
    # Load records for this record set
    records = dataset.records(record_set=record_set_id)
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set {record_set_id} with shape {df.shape}")

# Show columns for the first record set (if present)
if record_sets:
    first_record_set = record_sets[0]
    print(f"Columns of first record set ({first_record_set}):")
    print(dataframes[first_record_set].columns.tolist())
    display(dataframes[first_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
import numpy as np

# We must select a record set and numeric field to analyze
# Get the first DataFrame and attempt to pick a numeric column for demonstration
if record_sets:
    rs_id = record_sets[0]  # The first found record set
    df = dataframes[rs_id]
    # Select a numeric column if present
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_columns:
        numeric_field = numeric_columns[0]
        print(f"Using numeric field for analysis: {numeric_field}")
        threshold = df[numeric_field].quantile(0.75)  # Filter top 25% as example
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Pick a categorical/group field for group by (if present)
        # Avoid grouping by the numeric field selected, pick another column
        group_fields = [col for col in df.columns if col != numeric_field]
        group_field = None
        for candidate in group_fields:
            if df[candidate].dtype == 'object' and df[candidate].nunique() < 10:
                group_field = candidate
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping analysis.")
    else:
        print("No numeric fields found in the selected record set.")
else:
    print("No record sets found to analyze.")

## 5. Visualization
Visualize data distributions or the relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the selected numeric field (if available)
if record_sets and numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # Optional: Boxplot grouped by a group field if present
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the FAIR² Mental Health Survey Dataset from Croissant schema using `mlcroissant`.
- Basic overview and extraction of the data was performed using record set and field `@id`s for robust referencing.
- Simple exploratory and visualization steps revealed structural and value distributions for one of the key numeric fields. Further statistical or machine learning analyses can be performed using these DataFrames, referencing `@id` for all field, record set, and column identifiers in subsequent analyses.